# ATML PA2 — Task 5: Feedback source (RLVR vs RLAIF)
Kaggle: GPU T4 x2, Internet on. Nothing is trained: SFT is the untouched base policy; RLVR and RLAIF are the supplied
frozen LoRA adapters from the pinned course assets. Every step is resumable — after a disconnect, rerun from the top.

In [ ]:
REPO_URL = "https://github.com/danishrasool10/ATML-PA2.git"
WORK = "/kaggle/working/ATML-PA2"
CFG = "configs/feedback.yaml"

!test -d {WORK} || git clone -q {REPO_URL} {WORK}
!git -C {WORK} pull -q --ff-only && git -C {WORK} log --oneline -1
%cd {WORK}
!pip install -q -r requirements.txt
!python -c "import torch, transformers, peft, bitsandbytes; print(torch.__version__, transformers.__version__, peft.__version__, bitsandbytes.__version__, torch.cuda.device_count(), 'GPU(s)')"

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id="AbDu11aHHH/ATML-PA2-assets", repo_type="dataset",
                  revision="0b350481fb03f5525a35bcdec4131bd4fe487f98", local_dir=WORK,
                  allow_patterns=["data/**", "manifests/**", "checkpoints/rlvr_policy/**", "checkpoints/rlaif_policy/**",
                                  "checkpoints/rlvr_metadata.json", "checkpoints/rlaif_metadata.json"])

from common.data import load_yaml, read_jsonl
from task5_feedback.artifacts import verified_sha256
cfg = load_yaml(CFG)
for rel in (cfg["paths"]["gsm_eval"], cfg["paths"]["math_transfer_eval"], cfg["paths"]["task5_diagnostics"],
            f"{cfg['policies']['rlvr']}/adapter_model.safetensors", f"{cfg['policies']['rlaif']}/adapter_model.safetensors"):
    print(f"{rel:55s} {verified_sha256(rel)[:16]}")   # raises if a pinned release file was altered
print({k: len(read_jsonl(cfg["paths"][k])) for k in ("gsm_eval", "math_transfer_eval", "task5_diagnostics")})
!python -m pytest -q tests/test_task5_analysis.py

## Steps 1–3 — matched greedy generation, exact verifier, fixed pairwise judge
GPU0: GSM8K in-domain. GPU1: SVAMP transfer, then the 100-response diagnostic study. Each process generates the three
policies one at a time (CUDA cache cleared between policies), then loads the 4-bit judge. The second pass re-checks
everything and fills in whatever a worker missed; finished generations and cached judgments are skipped.

In [ ]:
%%bash
set -e
cd /kaggle/working/ATML-PA2
CFG=configs/feedback.yaml
run() { log=$1; shift; "$@" > "$log" 2>&1 || { tail -n 40 "$log"; exit 1; }; }
if [ "$(nvidia-smi -L | wc -l)" -ge 2 ]; then
  CUDA_VISIBLE_DEVICES=0 python -m task5_feedback.evaluate_math --config $CFG --dataset gsm > t5_gsm.log 2>&1 &
  CUDA_VISIBLE_DEVICES=1 sh -c "python -m task5_feedback.evaluate_math --config $CFG --dataset transfer && \
    python -m task5_feedback.score_perturbations --config $CFG" > t5_transfer_diag.log 2>&1 &
  wait
  grep -h -E "^\[(gsm|transfer):|Error|Traceback" t5_gsm.log t5_transfer_diag.log || true
fi
run t5_gsm_final.log python -m task5_feedback.evaluate_math --config $CFG --dataset gsm
run t5_transfer_final.log python -m task5_feedback.evaluate_math --config $CFG --dataset transfer
run t5_diag_final.log python -m task5_feedback.score_perturbations --config $CFG
tail -n 6 t5_gsm_final.log t5_transfer_final.log
tail -n 20 t5_diag_final.log

In [ ]:
import json
for d in ("gsm", "transfer"):
    m = json.load(open(f"{WORK}/results/task5_feedback/generation_manifest_{d}.json"))
    print(d, json.dumps(m["common"]["decoding"]), "prompt match:", m["common"]["release_prompt_match_rate"])
    for k, v in m["policies"].items():
        print(f"  {k:6s} acc={v['accuracy']:.3f} fmt={v['format_compliance']:.3f} tok={v['mean_response_tokens']:.1f} "
              f"trunc={v['truncation_rate']:.3f} wall={v['wall_s']:.0f}s peak={v['peak_vram_gib'] or 0:.2f}GiB")

## Final comparison

In [ ]:
!python -m task5_feedback.compare_feedback --config {CFG} > /dev/null
from IPython.display import Markdown
Markdown(open(f"{WORK}/results/task5_feedback/tables.md").read())

In [ ]:
!cp {WORK}/t5_*.log {WORK}/results/task5_feedback/ 2>/dev/null; cd {WORK} && zip -qr /kaggle/working/task5_results.zip results/task5_feedback outputs/task5_feedback
from IPython.display import FileLink, display
display(FileLink("task5_results.zip", result_html_prefix="Results: "))

Locally: unzip `task5_results.zip` at the repo root. `python -m task5_feedback.compare_feedback --config configs/feedback.yaml --recompute`
re-aggregates everything off-GPU from `outputs/task5_feedback/` (note `outputs/` is git-ignored; `results/` is not).